# GFED5.1 burned area on the GDSTEM grid

This notebook aggregates monthly GFED5.1 burned areas from aligned 0.25-degree pixels to the 0.5-degree GDSTEM grid. It:

1. reads unique TEM lower-left coordinates from the configured CSV;
2. finds one GFED file per year for 2002–2022;
3. checks coordinate axes, class variables, and area units;
4. sums 12 monthly areas and the four source pixels in each TEM cell;
5. tracks valid years, positive-fire years, means, and maxima for each of 16 GFED classes; and
6. writes a class-summary CSV and a sparse annual burned-area CSV.

Run the code cells from top to bottom. Edit the server paths in the first code cell before running. This is a proposed English rewrite; the original script is unchanged. The structure follows the local `fluxnet-gdstem-site-selection.ipynb` example: short English explanations, numbered steps, and executable code cells.


## Configuration and imports

Inputs are annual GFED files and a CSV containing `lon` and `lat`. The configured CSV is a potential-vegetation table, despite the legacy variable name `LULCC_CSV`. Coordinates identify TEM lower-left corners. Output filenames still contain `2002_2022`; update them if you change the period. Requires NumPy, pandas, xarray, and a NetCDF backend that supports groups (for example, netCDF4).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
# Edit paths for your server. Both output CSVs are written here.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')
# Create the output directory; existing output CSVs are overwritten when run.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GFED_DIR = Path('/quobyte/emoniergrp/TRENDY/input-files/source-input/GFED5/GFED5.1/Ecosystem')
GFED_GLOB = 'GFED5.1_ecosystem_*.nc'
START_YEAR = 2002
END_YEAR = 2022
N_YEARS = END_YEAR - START_YEAR + 1
# Only aligned 0.25-degree to 0.5-degree aggregation is supported.
GRID_RESOLUTION = 0.5
# Keep all 16 GFED categories separate; these IDs are not TEM communities.
GFED_NAMES = {
    1: 'Tundra',
    2: 'Sparse boreal forest',
    3: 'Boreal forest',
    4: 'Temperate grassland',
    5: 'Temperate shrubland',
    6: 'Temperate mosaic',
    7: 'Temperate forest',
    8: 'Tropical grassland',
    9: 'Tropical shrubland',
    10: 'Open savanna',
    11: 'Woody savanna',
    12: 'Tropical forest',
    13: 'Other',
    14: 'Cropland',
    15: 'Peat',
    16: 'Deforestation',
}
# The supplied script reports monthly accumulated areas with units='m2 per month'.
# Use m2 for that interpretation: sum monthly areas, then divide by 1,000,000.
# This overrides every variable units attribute; recheck it for other inputs.
GFED_UNITS_OVERRIDE = 'm2'
# The supplied sample ran south to north: -89.875 to +89.875 degrees.
# Fallback assumes all years have that order and west-to-east longitude.
# Constructed coordinates cannot detect changes in rejected actual coordinates.
GFED_LAT_FALLBACK = 'south_to_north'
# Read lon/lat from this configured potential-vegetation CSV.
# The code does not require a Stage 1 output; it only uses these two columns.
LULCC_CSV = Path('/quobyte/emoniergrp/emonier/gdstem/gdstem-preprocessing/static-land/static-data/gdstem-potveg-maxcohort.csv')
# These filenames retain the original 2002_2022 label; update if years change.
OUT_SUMMARY = OUTPUT_DIR / '02_gfed_by_class_summary_2002_2022.csv'
OUT_ANNUAL = OUTPUT_DIR / '02_gfed_annual_nonzero_2002_2022.csv'
R_EARTH_KM = 6371.0088


## 1. Read the TEM grid and find annual files

Read CSV coordinates in chunks, normalize longitudes to [-180, 180), remove duplicates, and check 0.5-degree alignment. Compute full geographic cell areas, rather than land/cohort areas. Each requested year must have exactly one file.


In [ ]:
def gridcell_area_km2(lat_centers, resolution):
    """Calculate full spherical grid-cell area in km2 from latitude centers.
    Area = R**2 * longitude_width_radians * (sin(north) - sin(south))."""
    lat = np.asarray(lat_centers, dtype=float)
    lo = np.deg2rad(np.maximum(lat - resolution / 2, -90))
    hi = np.deg2rad(np.minimum(lat + resolution / 2, 90))
    return R_EARTH_KM ** 2 * np.deg2rad(resolution) * (np.sin(hi) - np.sin(lo))

def tem_area_km2(lower_lat, resolution):
    """Convert TEM lower-left latitudes to centers, then calculate full cell areas."""
    return gridcell_area_km2(np.asarray(lower_lat) + resolution / 2, resolution)

def normalize_lon(lon):
    """Convert longitude degrees to the interval [-180, 180)."""
    return (np.asarray(lon, dtype=float) + 180.0) % 360.0 - 180.0

def find_files():
    """Find exactly one annual GFED file for every requested year."""
    found = sorted(GFED_DIR.glob(GFED_GLOB))
    if not found:
        raise FileNotFoundError(f'No files matching {GFED_DIR / GFED_GLOB}; check GFED_DIR and GFED_GLOB.')
    files_by_year = {}
    for path in found:
        year_text = path.stem[-4:]
        if path.suffix != '.nc' or len(year_text) != 4 or (not year_text.isdecimal()):
            continue
        year = int(year_text)
        if START_YEAR <= year <= END_YEAR:
            if year in files_by_year:
                raise ValueError(f'Duplicate GFED year {year}: {path} and {files_by_year[year]}')
            files_by_year[year] = path
    missing_years = []
    for year in range(START_YEAR, END_YEAR + 1):
        if year not in files_by_year:
            missing_years.append(year)
    if missing_years:
        raise ValueError(f'Missing GFED files for years {missing_years}')
    return files_by_year

def read_tem_grid():
    """Read unique lower-left TEM coordinates from the configured CSV in chunks."""
    if not LULCC_CSV.is_file():
        raise FileNotFoundError(f'TEM coordinate CSV not found: {LULCC_CSV}')
    # Deduplicate each chunk, then deduplicate again across the entire CSV.
    collected = []
    for chunk in pd.read_csv(LULCC_CSV, usecols=['lon', 'lat'], chunksize=100000):
        collected.append(chunk.drop_duplicates())
    grid = pd.concat(collected, ignore_index=True).drop_duplicates().reset_index(drop=True)
    grid['lon'] = np.round(normalize_lon(grid['lon']), 6)
    grid['lat'] = np.round(grid['lat'], 6)
    grid = grid.drop_duplicates().sort_values(['lat', 'lon']).reset_index(drop=True)
    for name in ('lon', 'lat'):
        if not np.allclose(grid[name] / GRID_RESOLUTION, np.rint(grid[name] / GRID_RESOLUTION), atol=1e-06):
            raise ValueError(f'TEM {name} values are not 0.5° left-corner ticks; inspect LULCC')
    grid['grid_area_km2'] = tem_area_km2(grid['lat'], GRID_RESOLUTION)
    print(f'{len(grid):,} unique TEM grid cells; lat/lon sample:')
    print(grid.head(3).to_string(index=False))
    return grid


## 2. Read GFED coordinates, classes, and units

Try coordinates in the burned-area group first, then in the root group. Class IDs 01–16 identify ecosystems, not months.

**Inherited coordinate limitation:** the original latitude extent test rejects valid ±89.875-degree centers. The configured `south_to_north` fallback reconstructs a global 720 × 1440 grid. It cannot verify actual coordinate orientation across years when those coordinates are rejected. This rewrite retains that behavior.

**Inherited unit assumption:** `GFED_UNITS_OVERRIDE = "m2"` interprets the supplied monthly values as accumulated areas. It overrides all input units attributes. The underlying NetCDF files were not available for independent verification here.


In [ ]:
def coordinate_1d(arr, is_lat, count):
    """Validate a regular geographic axis; preserve its original array order.
    The original +/-89.9 latitude-extent check is retained. It rejects
    +/-89.875 centers, so the current GFED files use the configured fallback."""
    x = np.asarray(arr, dtype=float)
    if x.ndim == 2:
        if is_lat and x.shape[0] == count and np.allclose(x, x[:, :1], equal_nan=True):
            x = x[:, 0]
        elif not is_lat and x.shape[1] == count and np.allclose(x, x[:1, :], equal_nan=True):
            x = x[0, :]
    if x.ndim != 1 or len(x) != count or (not np.all(np.isfinite(x))):
        return None
    if is_lat and (np.min(x) < -90 or np.max(x) > 90):
        return None
    if not is_lat and (np.min(x) < -180 or np.max(x) > 360):
        return None
    # Sorting checks spacing only; return the original north/south order.
    values = np.sort(np.unique(x))
    if len(values) != count or not np.allclose(np.diff(values), 0.25, atol=0.0001):
        return None
    # Original extent threshold retained; see the documented fallback limitation.
    if is_lat and (np.min(x) > -89.9 or np.max(x) < 89.9):
        return None
    if not is_lat and np.max(x) - np.min(x) < 359:
        return None
    return x

def find_coordinate_pair(dataset, nlat, nlon):
    """Look for validated latitude and longitude axes in one NetCDF group."""
    coordinates = {}
    for (
        axis,
        possible_names,
        length,
    ) in (('lat', ('lat', 'latitude'), nlat), ('lon', ('lon', 'longitude'), nlon)):
        coordinates[axis] = None
        for name in possible_names:
            if name in dataset:
                values = coordinate_1d(dataset[name].values, axis == 'lat', length)
                if values is not None:
                    coordinates[axis] = values
                    break
    return (coordinates['lon'], coordinates['lat'])

def read_coordinates(ds, fname, nlat, nlon):
    """Read group/root coordinates, or use the explicitly configured fallback."""
    (lon, lat) = find_coordinate_pair(ds, nlat, nlon)
    if lon is not None and lat is not None:
        return (lon, lat)
    with xr.open_dataset(fname) as root:
        (lon, lat) = find_coordinate_pair(root, nlat, nlon)
        if lon is not None and lat is not None:
            return (lon, lat)
    if GFED_LAT_FALLBACK not in ('north_to_south', 'south_to_north'):
        raise ValueError(f'Cannot find verified geographic GFED coordinates in {fname}. Inspect file orientation before setting GFED_LAT_FALLBACK in the configuration section.')
    if (nlat, nlon) != (720, 1440):
        raise ValueError(f'Fallback only supports 720x1440, found {nlat}x{nlon}')
    lon = -179.875 + 0.25 * np.arange(nlon)
    if GFED_LAT_FALLBACK == 'north_to_south':
        lat = 89.875 - 0.25 * np.arange(nlat)
    else:
        lat = -89.875 + 0.25 * np.arange(nlat)
    print('WARNING: Using manually configured 0.25-degree coordinate fallback:', GFED_LAT_FALLBACK)
    return (lon, lat)

def get_units(da):
    """Return a supported area-unit code, preferring the configured override."""
    unit = GFED_UNITS_OVERRIDE if GFED_UNITS_OVERRIDE is not None else da.attrs.get('units')
    if unit is None:
        raise ValueError(f'{da.name} missing units attribute; inspect the file and set a verified GFED_UNITS_OVERRIDE in the configuration section')
    u = str(unit).lower().replace(' ', '').replace('²', '2')
    codes = {
        'm2': 'm2',
        'm^2': 'm2',
        'm**2': 'm2',
        'km2': 'km2',
        'km^2': 'km2',
        'km**2': 'km2',
        'ha': 'ha',
        'hectares': 'ha',
        'hectare': 'ha',
        '1': 'fraction',
        'fraction': 'fraction',
        'unitless': 'fraction',
        '%': 'percent',
        'percent': 'percent',
        'percentage': 'percent',
    }
    if u not in codes:
        raise ValueError(f'Unrecognized units {unit!r} for {da.name}, attrs={da.attrs}')
    return codes[u]

def get_category_variables(dataset, filename):
    """Read class IDs from BA_01_Tundra-style names without a regular expression."""
    variables = {}
    for name in dataset.data_vars:
        parts = name.split('_')
        if len(parts) < 2 or parts[0] != 'BA':
            continue
        if len(parts[1]) != 2 or not parts[1].isdecimal():
            continue
        class_id = int(parts[1])
        if class_id in variables:
            raise ValueError(f'Duplicate GFED category {class_id} in {filename}')
        variables[class_id] = name
    if set(variables) != set(range(1, 17)):
        raise ValueError(f'Expected 16 BA_XX variables in {filename}; found {sorted(variables)}')
    return variables


## 3. Map source pixels and calculate annual areas

A table merge assigns GFED pixel centers to TEM lower-left corners. Each target cell must receive exactly four source pixels. Monthly areas are summed and converted to km²; no multiplication by days or seconds is applied. Fraction and percent inputs use full source-pixel areas.

`groupby().sum()` replaces weighted `np.bincount`. All four source pixels must each have 12 finite monthly values for a TEM cell-year to be valid. The code checks that there are 12 slices; it does not independently verify January–December timestamps. Memory use includes a full year for one category, temporary arrays, and the spatial matching tables.


In [ ]:
def make_spatial_mapping(grid, lon, lat):
    """Match GFED pixel centers to TEM lower-left corners using a table merge."""
    lon_lower = np.round(
        np.floor(normalize_lon(lon) / GRID_RESOLUTION + 1e-10) * GRID_RESOLUTION,
        6,
    )
    lat_lower = np.round(np.floor(np.asarray(lat) / GRID_RESOLUTION + 1e-10) * GRID_RESOLUTION, 6)
    # Flatten in latitude-then-longitude order to match the burned-area arrays.
    (source_lon, source_lat) = np.meshgrid(lon_lower, lat_lower)
    source_cells = pd.DataFrame({
        'lon': source_lon.ravel(),
        'lat': source_lat.ravel(),
        'source_index': np.arange(source_lon.size),
    })
    target_cells = grid[['lon', 'lat']].copy()
    target_cells['target_id'] = np.arange(len(grid))
    matches = source_cells.merge(
        target_cells,
        on=['lon', 'lat'],
        how='inner',
        validate='many_to_one',
    )
    matches = matches.sort_values('source_index')
    source_indices = matches['source_index'].to_numpy()
    target_ids = matches['target_id'].to_numpy()
    # Count matched source pixels for every target cell, including unmatched cells.
    source_counts = matches.groupby('target_id').size().reindex(
        range(len(grid)),
        fill_value=0,
    ).to_numpy()
    print(
        'GFED source-cell counts per TEM grid cell:',
        pd.Series(source_counts).value_counts().sort_index().to_dict(),
    )
    if np.any(source_counts != 4):
        raise ValueError('Not all TEM cells have 4 GFED source pixels. Inspect coordinate bounds and aggregation convention.')
    # Repeat each latitude-band area across all source longitudes.
    latitude_areas = gridcell_area_km2(lat, 0.25)
    source_areas = np.repeat(latitude_areas, len(lon))[source_indices]
    return (source_indices, target_ids, source_areas, source_counts)

def annual_source_ba(da, source_indices, area_source, units, nlat, nlon):
    """Sum 12 monthly slices and convert source burned areas to annual km2.
    A source pixel is valid only if all 12 monthly values are finite.
    One full year for one category is loaded at a time.
    Monthly accumulated areas are summed without multiplying by days or seconds."""
    spatial_dims = [d for d in da.dims if da.sizes[d] == nlat and d.lower() in ('lat', 'latitude')]
    lon_dims = [d for d in da.dims if da.sizes[d] == nlon and d.lower() in ('lon', 'longitude')]
    if len(spatial_dims) != 1 or len(lon_dims) != 1:
        raise ValueError(f'Unexpected GFED BA dims: {da.name}: {da.dims}, {da.shape}')
    (latd, lond) = (spatial_dims[0], lon_dims[0])
    temporal = [d for d in da.dims if d not in (latd, lond)]
    if len(temporal) != 1 or da.sizes[temporal[0]] != 12:
        raise ValueError(f'Expected 12 monthly slices, got {da.name} {da.sizes}')
    # Load one category for all 12 months, then retain mapped source pixels.
    v = np.asarray(da.transpose(temporal[0], latd, lond).values, dtype=np.float64)
    v = v.reshape(12, -1)[:, source_indices]
    # Any missing monthly value invalidates this source pixel for the year.
    valid = np.isfinite(v).all(axis=0)
    # Retain the original negative-value tolerance; tiny negatives are not clipped.
    if np.any((v < -1e-09) & np.isfinite(v)):
        raise ValueError(f'Negative GFED burned area in {da.name}; inspect units/fill values')
    # Replace missing values for arithmetic only; validity is checked separately.
    total = np.where(np.isfinite(v), v, 0).sum(axis=0)
    # Convert annual accumulated area to km2; never multiply by timestep seconds.
    if units == 'm2':
        total /= 1000000.0
    elif units == 'ha':
        total /= 100.0
    elif units == 'fraction':
        total *= area_source
    elif units == 'percent':
        total *= area_source / 100.0
    return (total, valid)

def aggregate_to_tem(burned_area, valid_source, target_ids, source_counts, number_of_cells):
    """Sum source areas by TEM cell and require all four source pixels to be valid."""
    source_table = pd.DataFrame({
        'target_id': target_ids,
        'burned_area_km2': burned_area,
        'valid_source': valid_source.astype(int),
    })
    # Sum areas and valid-pixel counts separately for each TEM cell.
    totals = source_table.groupby('target_id')[['burned_area_km2', 'valid_source']].sum()
    totals = totals.reindex(range(number_of_cells), fill_value=0)
    cell_area = totals['burned_area_km2'].to_numpy()
    valid_cells = totals['valid_source'].to_numpy() == source_counts
    return (cell_area, valid_cells)


## 4. Initialize the grid and summary arrays

For each category and TEM cell, store accumulated area, maximum annual area, valid-year count, and positive-fire-year count. Valid zero-fire years contribute to the mean denominator. Counts retain the original uint8 type, which can represent up to 255 years.


In [ ]:
grid = read_tem_grid()
files = find_files()
N = len(grid)
# Track area sum, maximum, valid-year count, and positive-fire-year count.
# Counts retain uint8 from the original: appropriate for the current 21 years.
accum = {}
for gfed_id in range(1, 17):
    accum[gfed_id] = {
        'sum': np.zeros(N),
        'max': np.zeros(N),
        'nvalid': np.zeros(N, dtype=np.uint8),
        'nfire': np.zeros(N, dtype=np.uint8),
    }
first_ann = True
# Reuse the first mapping and compare returned coordinates in later years.
reference_coords = None
reference_units = {}


## 5. Process each year and each GFED class

Process one annual NetCDF at a time and one class at a time. Only valid cell-years update summary statistics. Write annual rows only when burned area is strictly positive. Running this workflow overwrites the existing annual output on its first write, then appends subsequent years.


In [ ]:
for year in range(START_YEAR, END_YEAR + 1):
    f = files[year]
    print('Processing GFED', year, f, flush=True)
    with xr.open_dataset(f, group='burned_area_partitioning') as ds:
        vars_by_id = get_category_variables(ds, f)
        first = ds[vars_by_id[1]]
        names = [d for d in first.dims if d.lower() in ('lat', 'latitude')]
        names2 = [d for d in first.dims if d.lower() in ('lon', 'longitude')]
        if len(names) != 1 or len(names2) != 1:
            raise ValueError(f'Expected lat/lon dimension names; found {first.dims}')
        (nlat, nlon) = (first.sizes[names[0]], first.sizes[names2[0]])
        (lon, lat) = read_coordinates(ds, f, nlat, nlon)
        if reference_coords is None:
            (ref_lon, ref_lat) = (lon, lat)
            (src_index, target_id, source_area, source_count) = make_spatial_mapping(grid, lon, lat)
            # Reuse the first mapping and compare returned coordinates in later years.
            reference_coords = (ref_lon, ref_lat)
        elif not (np.allclose(lon, reference_coords[0]) and np.allclose(lat, reference_coords[1])):
            raise ValueError(f'GFED spatial coordinates changed in {f}')
        annual_rows = []
        for gfed_id in range(1, 17):
            da = ds[vars_by_id[gfed_id]]
            # With an override, this comparison cannot detect changes in input units attributes.
            units = get_units(da)
            if gfed_id in reference_units and units != reference_units[gfed_id]:
                raise ValueError(f'GFED units changed for class {gfed_id} in {f}')
            reference_units[gfed_id] = units
            (ba_src, valid_src) = annual_source_ba(da, src_index, source_area, units, nlat, nlon)
            # Require four source pixels with twelve finite months each.
            (ba_by_cell, valid_tgt) = aggregate_to_tem(
                ba_src,
                valid_src,
                target_id,
                source_count,
                N,
            )
            state = accum[gfed_id]
            # Include valid zero-fire years in the denominator of the annual mean.
            state['sum'][valid_tgt] += ba_by_cell[valid_tgt]
            state['max'][valid_tgt] = np.maximum(state['max'][valid_tgt], ba_by_cell[valid_tgt])
            state['nvalid'] += valid_tgt.astype(np.uint8)
            # Only valid positive-fire cell-years enter the sparse annual output CSV.
            active = valid_tgt & (ba_by_cell > 0)
            state['nfire'] += active.astype(np.uint8)
            inds = np.flatnonzero(active)
            if len(inds):
                annual_rows.append(pd.DataFrame({
                    'lon': grid['lon'].to_numpy()[inds],
                    'lat': grid['lat'].to_numpy()[inds],
                    'year': year,
                    'gfed_id': gfed_id,
                    'burned_area_km2': ba_by_cell[inds],
                }))
        if annual_rows:
            annual_part = pd.concat(annual_rows, ignore_index=True)
            annual_part.to_csv(
                OUT_ANNUAL,
                mode='w' if first_ann else 'a',
                header=first_ann,
                index=False,
            )
            first_ann = False


## 6. Write class summaries

Mean annual burned area equals the sum over valid years divided by the number of valid years, including valid zero-fire years. No valid years produce NaN means/maxima and zero sums/counts.

`gfed_mean_grid_fraction` divides mean annual burned area by the **full geographic TEM cell area**. It is a fraction, not a percentage, land-only fraction, or class-specific fraction. Repeated burning is not clipped. Running this step overwrites the class-summary output, then appends the remaining classes.


In [ ]:
# Write an empty annual table with headers if no positive-fire records exist.
if first_ann:
    pd.DataFrame(columns=['lon', 'lat', 'year', 'gfed_id', 'burned_area_km2']).to_csv(
        OUT_ANNUAL,
        index=False,
    )
print('Writing GFED class summaries...', flush=True)
for k in range(1, 17):
    state = accum[k]
    nvalid = state['nvalid']
    # No valid years: mean and maximum remain NaN; sum retains its original zero.
    mean = np.full(N, np.nan)
    # Mean annual burned area = sum of valid annual areas / number of valid years.
    has_data = nvalid > 0
    mean[has_data] = state['sum'][has_data] / nvalid[has_data]
    part = grid[['lon', 'lat', 'grid_area_km2']].copy()
    part['gfed_id'] = k
    part['gfed_name'] = GFED_NAMES[k]
    part['gfed_ba_sum_km2'] = state['sum']
    part['gfed_mean_ba_km2'] = mean
    part['gfed_max_annual_ba_km2'] = np.where(nvalid > 0, state['max'], np.nan)
    part['gfed_valid_years'] = nvalid
    part['gfed_nonzero_years'] = state['nfire']
    # Divide by the full geographic TEM cell area, including land and ocean.
    # This is a fraction, not percent or class-covered fraction; do not clip to 1.
    part['gfed_mean_grid_fraction'] = mean / part['grid_area_km2']
    part.to_csv(OUT_SUMMARY, index=False, mode='w' if k == 1 else 'a', header=k == 1)
    print(f' class {k:02d}, cells with complete {N_YEARS} years: {int((nvalid == N_YEARS).sum()):,}')
print('Saved', OUT_SUMMARY)
print('Saved', OUT_ANNUAL)


## Output tables

| File | Contents |
| --- | --- |
| `02_gfed_by_class_summary_2002_2022.csv` | One row per TEM cell and GFED class: coordinates, full grid area, class ID/name, burned-area sum, annual mean/maximum, valid-year count, positive-fire-year count, and mean grid fraction. |
| `02_gfed_annual_nonzero_2002_2022.csv` | Valid positive-fire records: `lon`, `lat`, `year`, `gfed_id`, `burned_area_km2`. |

Absence from the annual table can mean either zero fire or invalid data; the sparse table alone does not distinguish them. Summary valid-year counts include zero-fire years. All 16 GFED classes remain separate; no GFED-to-TEM community crosswalk is applied. Outputs have the same names, columns, and class/year writing order as the original script.
